# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Lane: Refresh / Content Opportunity Scoring

This notebook turns the validated Week-5 model output into a practical,
human-reviewed content action playbook.

The playbook uses March 2026 performance signals and the validated
Week-5 Logistic Regression model.

The purpose is decision-support.

The recommendations are not guarantees that a content change will improve
future performance, and they are not intended to automate publishing or
content changes.

In [4]:
import pandas as pd
import numpy as np

from pathlib import Path

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Libraries loaded.")

Libraries loaded.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue is based on the validated Week-5 model output.

Each observation receives:

- a model probability;
- a priority score;
- a reason code based on observable March signals;
- an archetype;
- a recommended action.

The ranking is intended to help a human reviewer decide what to inspect first.

A high ranking does not mean that the page should automatically be changed.

#### Login

In [5]:
HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face login complete.")

Hugging Face login complete.


#### Download March and April

In [6]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March file ready.")
print("April file ready.")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March file ready.
April file ready.


#### Load March

In [7]:
feature_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_ai",
    "scroll_events"
]

march_raw = pd.read_parquet(
    march_file,
    columns=feature_columns
)

print("March rows:", len(march_raw))
print(
    "March date range:",
    march_raw["report_date"].min(),
    "to",
    march_raw["report_date"].max()
)

March rows: 9841378
March date range: 2026-03-01 to 2026-03-31


#### Aggregate March

In [8]:
march_features = (
    march_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean"),
        march_sessions=("ga4_sessions", "sum"),
        march_engaged_sessions=("ga4_engaged_sessions", "sum"),
        march_organic_sessions=("sessions_organic", "sum"),
        march_ai_sessions=("sessions_ai", "sum"),
        march_scroll_events=("scroll_events", "sum")
    )
)

march_features["march_ctr"] = np.where(
    march_features["march_impressions"] > 0,
    march_features["march_clicks"] /
    march_features["march_impressions"],
    np.nan
)

march_features["march_engagement_rate"] = np.where(
    march_features["march_sessions"] > 0,
    march_features["march_engaged_sessions"] /
    march_features["march_sessions"],
    np.nan
)

print("Aggregated March rows:", len(march_features))

Aggregated March rows: 331437


#### load April outcome

In [9]:
april_raw = pd.read_parquet(
    april_file,
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ga4_sessions",
        "ga4_engaged_sessions"
    ]
)

print("April rows:", len(april_raw))

April rows: 10424730


#### Aggregate April

In [10]:
april_outcome = (
    april_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum"),
        april_sessions=("ga4_sessions", "sum"),
        april_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)

print("Aggregated April rows:", len(april_outcome))

Aggregated April rows: 362172


#### Create modeling dataset

In [11]:
model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["future_decline"] = np.where(
    model_df["march_clicks"] > 0,
    (
        model_df["april_clicks"] <
        model_df["march_clicks"]
    ).astype(int),
    np.nan
)

model_df = model_df.dropna(
    subset=["future_decline"]
).copy()

model_df["future_decline"] = (
    model_df["future_decline"]
    .astype(int)
)

print("Modeling rows:", len(model_df))
print(
    "Future decline rate:",
    round(model_df["future_decline"].mean(), 4)
)

Modeling rows: 68837
Future decline rate: 0.6552


#### Same Week-5 features

In [12]:
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_sessions",
    "march_engaged_sessions",
    "march_organic_sessions",
    "march_ai_sessions",
    "march_scroll_events",
    "march_ctr",
    "march_engagement_rate"
]

X = model_df[feature_cols].copy()
y = model_df["future_decline"].copy()
groups = model_df["client_hash_id"].copy()

print("Feature matrix:", X.shape)

Feature matrix: (68837, 10)


#### Reproduce Week-5 grouped split

In [13]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_rows = model_df.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print(
    "Train clients:",
    groups.iloc[train_idx].nunique()
)
print(
    "Test clients:",
    groups.iloc[test_idx].nunique()
)

Train rows: 63775
Test rows: 5062
Train clients: 35
Test clients: 9


#### Train the same Week-5 model

In [14]:
model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model.fit(
    X_train,
    y_train
)

model_probability = (
    model.predict_proba(X_test)[:, 1]
)

print("Week-5 model reproduced.")

Week-5 model reproduced.


#### Build ranked queue

In [15]:
queue = test_rows[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_avg_position",
        "march_sessions",
        "march_engagement_rate",
        "march_organic_sessions",
        "march_ai_sessions",
        "march_scroll_events"
    ]
].copy()

queue["model_probability"] = model_probability

#### Reason codes

In [16]:
queue["reason_code"] = "monitor"

mask = (
    (queue["march_impressions"] >= 500) &
    (queue["march_ctr"] < 0.005)
)

queue.loc[
    mask,
    "reason_code"
] = "low_ctr_visible_page"

mask = (
    (queue["reason_code"] == "monitor") &
    (queue["march_sessions"] >= 30) &
    (queue["march_engagement_rate"].notna()) &
    (queue["march_engagement_rate"] < 0.30)
)

queue.loc[
    mask,
    "reason_code"
] = "low_engagement_visible_page"

mask = (
    (queue["reason_code"] == "monitor") &
    (queue["march_impressions"] >= 100) &
    (queue["march_clicks"] > 0)
)

queue.loc[
    mask,
    "reason_code"
] = "demand_present"

mask = (
    (queue["reason_code"] == "monitor") &
    (queue["march_impressions"] >= 500)
)

queue.loc[
    mask,
    "reason_code"
] = "visible_page"

print(
    queue["reason_code"].value_counts()
)

reason_code
demand_present                 2561
low_ctr_visible_page           1486
monitor                         886
low_engagement_visible_page     129
Name: count, dtype: int64


#### Archetypes

In [17]:
queue["archetype"] = "low_evidence"

queue.loc[
    (
        (queue["march_impressions"] >= 500) &
        (queue["march_ctr"] < 0.005)
    ),
    "archetype"
] = "high_visibility_low_ctr"

queue.loc[
    (
        (queue["march_sessions"] >= 30) &
        (queue["march_engagement_rate"].notna()) &
        (queue["march_engagement_rate"] < 0.30)
    ),
    "archetype"
] = "high_traffic_low_engagement"

queue.loc[
    (
        (queue["march_impressions"] >= 100) &
        (queue["march_clicks"] > 0) &
        (queue["archetype"] == "low_evidence")
    ),
    "archetype"
] = "demand_with_clicks"

queue.loc[
    (
        (queue["march_impressions"] >= 500) &
        (queue["archetype"] == "low_evidence")
    ),
    "archetype"
] = "visible_without_strong_issue"

print(
    queue["archetype"].value_counts()
)

archetype
demand_with_clicks             2561
high_visibility_low_ctr        1473
low_evidence                    886
high_traffic_low_engagement     142
Name: count, dtype: int64


#### Priority score

In [18]:
queue["priority_score"] = (
    queue["model_probability"] * 100
)

queue = queue.sort_values(
    by=[
        "priority_score",
        "march_impressions"
    ],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = (
    queue.index + 1
)

print("Ranked queue created.")
display(
    queue.head(20)
)

Ranked queue created.


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,march_sessions,march_engagement_rate,march_organic_sessions,march_ai_sessions,march_scroll_events,model_probability,reason_code,archetype,priority_score,rank
0,client_f623b01661d4bfe4,content_b589c670c71a5db2,1,1,1.0,58.0,1.0,0.0,2.0,0.0,0.0,0.999999,monitor,low_evidence,99.999896,1
1,client_cd12bcfd98942aa1,content_9a5496694a583150,1,1,1.0,58.0,4.0,0.0,1.0,0.0,0.0,0.999999,monitor,low_evidence,99.999896,2
2,client_f623b01661d4bfe4,content_507653893440072c,1,1,1.0,45.0,0.0,NaN,0.0,0.0,0.0,0.999999,monitor,low_evidence,99.999896,3
3,client_f623b01661d4bfe4,content_b11dddc3f12c8b25,1,1,1.0,47.0,1.0,0.0,0.0,0.0,0.0,0.999999,monitor,low_evidence,99.999896,4
4,client_cd12bcfd98942aa1,content_45b21e26eaa53525,1,1,1.0,10.0,1.0,0.0,1.0,0.0,0.0,0.999999,monitor,low_evidence,99.999892,5
5,client_f623b01661d4bfe4,content_7d9c2bb43460c94d,1,1,1.0,9.0,1.0,0.0,2.0,0.0,0.0,0.999999,monitor,low_evidence,99.999891,6
6,client_f623b01661d4bfe4,content_f4887eb4e7975c51,1,1,1.0,7.0,1.0,0.0,1.0,0.0,0.0,0.999999,monitor,low_evidence,99.999891,7
7,client_f623b01661d4bfe4,content_4fa145e34d62c97a,1,1,1.0,6.0,1.0,0.0,1.0,0.0,0.0,0.999999,monitor,low_evidence,99.999891,8
8,client_f623b01661d4bfe4,content_38635b8d3830a3ca,1,1,1.0,1.0,0.0,NaN,0.0,0.0,0.0,0.999999,monitor,low_evidence,99.999891,9
9,client_f623b01661d4bfe4,content_6bd6b23eb02abe1b,1,1,1.0,0.0,0.0,NaN,0.0,0.0,0.0,0.999999,monitor,low_evidence,99.999891,10


#### Archetype → action

In [19]:
action_map = {
    "high_visibility_low_ctr":
        "review_title_snippet_intent",
    "high_traffic_low_engagement":
        "review_content_experience",
    "demand_with_clicks":
        "review_refresh_opportunity",
    "visible_without_strong_issue":
        "human_review",
    "low_evidence":
        "monitor"
}

queue["recommended_action"] = (
    queue["archetype"]
    .map(action_map)
    .fillna("human_review")
)

display(
    queue[
        [
            "rank",
            "model_probability",
            "priority_score",
            "archetype",
            "reason_code",
            "recommended_action"
        ]
    ].head(20)
)

,rank,model_probability,priority_score,archetype,reason_code,recommended_action
0,1,0.999999,99.999896,low_evidence,monitor,monitor
1,2,0.999999,99.999896,low_evidence,monitor,monitor
2,3,0.999999,99.999896,low_evidence,monitor,monitor
3,4,0.999999,99.999896,low_evidence,monitor,monitor
4,5,0.999999,99.999892,low_evidence,monitor,monitor
5,6,0.999999,99.999891,low_evidence,monitor,monitor
6,7,0.999999,99.999891,low_evidence,monitor,monitor
7,8,0.999999,99.999891,low_evidence,monitor,monitor
8,9,0.999999,99.999891,low_evidence,monitor,monitor
9,10,0.999999,99.999891,low_evidence,monitor,monitor


The queue is ranked by the measured Week-5 model probability.

The reason code and archetype provide human-readable context for the ranking.

The action is a recommendation for review, not an automatic instruction to change content.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

The playbook is intended for a content or SEO reviewer who needs to prioritize
which observations deserve investigation first.

The output can help answer:

- Which observations should be reviewed first?
- What measurable signal caused the observation to enter the queue?
- What type of review may be appropriate?

### Limits

The model is based on March 2026 signals and the validation setup used in Week 5.

The measured future outcome was a decline in April clicks relative to March clicks
for observations with March click demand.

This is an observed outcome definition, not a direct measure of whether a content
refresh would succeed.

The model does not establish causality.

It also does not contain information about editorial quality, search intent,
business value, technical SEO issues, or the actual reason a page changed.

Therefore the queue should be treated as directional decision-support.

It should not automatically publish, rewrite, delete, redirect, or refresh content.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every recommended action requires human review.

Before acting on an observation, the reviewer should check:

1. Search intent still matches the page.
2. The page is actually relevant to the target topic.
3. The observed signal is not explained by an obvious measurement issue.
4. Seasonality or temporary demand changes are considered.
5. The proposed content change is appropriate for the page.
6. The recommendation has enough evidence to justify the work.

### No-go cases

The system should not automatically:

- publish content changes;
- rewrite a page;
- delete content;
- redirect a page;
- change canonicalization;
- change internal links;
- change metadata;
- make business-critical decisions;
- treat model probability as proof of a content problem;
- treat a high score as proof that a refresh will improve performance.

Low-evidence observations should normally remain in monitoring rather than being
automatically escalated.

The final decision remains with a human reviewer.

#### human review status

In [20]:
queue["human_review_required"] = True

queue["automation_allowed"] = False

print(
    "Automatic content actions allowed:",
    queue["automation_allowed"].any()
)

print(
    "Human review required:",
    queue["human_review_required"].all()
)

Automatic content actions allowed: False
Human review required: True


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The playbook should not be assumed to remain valid forever.

The following signals should trigger review of the model or rules:

- the distribution of model probabilities changes substantially;
- the observed decline rate changes substantially;
- precision or recall falls materially on a newly evaluated period;
- the ranking becomes dominated by low-evidence observations;
- new data sources change the meaning or availability of existing features;
- the relationship between March signals and later outcomes changes;
- the business definition of a useful content action changes.

A retrain should be considered when new labeled periods are available and the
model has enough fresh observations to support a meaningful validation.

Retraining should not happen only because the model can be made more complex.

The same leakage checks and honest validation design should be repeated after retraining.

#### Simple monitoring snapshot

In [21]:
monitoring_snapshot = pd.DataFrame(
    {
        "metric": [
            "queue_rows",
            "mean_model_probability",
            "high_priority_share",
            "low_evidence_share",
            "human_review_required"
        ],
        "value": [
            len(queue),
            queue["model_probability"].mean(),
            (
                queue["model_probability"] >= 0.70
            ).mean(),
            (
                queue["archetype"] == "low_evidence"
            ).mean(),
            True
        ]
    }
)

display(
    monitoring_snapshot.round(4)
)

,metric,value
0,queue_rows,5062
1,mean_model_probability,0.668681
2,high_priority_share,0.120308
3,low_evidence_share,0.17503
4,human_review_required,True


#### Archetype monitoring

In [22]:
archetype_monitoring = (
    queue
    .groupby("archetype", as_index=False)
    .agg(
        observations=("content_hash_id", "count"),
        mean_probability=("model_probability", "mean"),
        mean_impressions=("march_impressions", "mean"),
        mean_clicks=("march_clicks", "mean")
    )
    .sort_values(
        "mean_probability",
        ascending=False
    )
)

display(
    archetype_monitoring.round(4)
)

,archetype,observations,mean_probability,mean_impressions,mean_clicks
3,low_evidence,886,0.7600,44.6445,1.2460
0,demand_with_clicks,2561,0.6554,541.7314,4.5748
1,high_traffic_low_engagement,142,0.6460,5453.0282,66.1831
2,high_visibility_low_ctr,1473,0.6390,2040.9925,4.3618


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked queue and monitoring summaries are exported so that the research paper
can reuse the measured outputs from this notebook.

The queue remains a generated data artifact and does not need to be committed to git.

The notebook is the reproducible source for regenerating it.

#### Export queue

In [23]:
output_dir = Path("work/outputs")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

queue_path = (
    output_dir /
    "w07_content_action_queue.csv"
)

queue.to_csv(
    queue_path,
    index=False
)

print("Queue saved:", queue_path)
print("Rows written:", len(queue))

Queue saved: work/outputs/w07_content_action_queue.csv
Rows written: 5062


#### Export archetype summary

In [24]:
archetype_path = (
    output_dir /
    "w07_archetype_action_summary.csv"
)

archetype_summary = (
    queue
    .groupby(
        [
            "archetype",
            "recommended_action",
            "reason_code"
        ],
        as_index=False
    )
    .agg(
        observations=("content_hash_id", "count"),
        mean_probability=("model_probability", "mean"),
        mean_priority_score=("priority_score", "mean")
    )
    .sort_values(
        "mean_probability",
        ascending=False
    )
)

archetype_summary.to_csv(
    archetype_path,
    index=False
)

print(
    "Archetype summary saved:",
    archetype_path
)

Archetype summary saved: work/outputs/w07_archetype_action_summary.csv


#### Export monitoring snapshot

In [25]:
monitoring_path = (
    output_dir /
    "w07_monitoring_snapshot.csv"
)

monitoring_snapshot.to_csv(
    monitoring_path,
    index=False
)

print(
    "Monitoring snapshot saved:",
    monitoring_path
)

Monitoring snapshot saved: work/outputs/w07_monitoring_snapshot.csv


#### Final export check

In [26]:
print("EXPORT CHECK")
print("=" * 50)

for path in [
    queue_path,
    archetype_path,
    monitoring_path
]:
    print(
        path,
        "->",
        path.exists()
    )

EXPORT CHECK
work/outputs/w07_content_action_queue.csv -> True
work/outputs/w07_archetype_action_summary.csv -> True
work/outputs/w07_monitoring_snapshot.csv -> True


### Decay / refresh insight

The April data provides a retrospective check of how March performance signals
related to the observed next-month click outcome.

This analysis is used to understand the measured relationship between observable
March conditions and later click decline.

It is not used to claim that a refresh caused or would prevent the decline.

In [27]:
decay_analysis = model_df[
    [
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_sessions",
        "march_engagement_rate",
        "future_decline"
    ]
].copy()

decay_analysis["visibility_band"] = pd.cut(
    decay_analysis["march_impressions"],
    bins=[-1, 99, 499, np.inf],
    labels=[
        "low_visibility",
        "medium_visibility",
        "high_visibility"
    ]
)

decay_summary = (
    decay_analysis
    .groupby(
        "visibility_band",
        observed=True
    )
    .agg(
        observations=("future_decline", "size"),
        observed_decline_rate=("future_decline", "mean")
    )
    .reset_index()
)

display(
    decay_summary.round(4)
)

,visibility_band,observations,observed_decline_rate
0,low_visibility,5175,0.8319
1,medium_visibility,12534,0.6903
2,high_visibility,51128,0.6287


The table above is a descriptive decay check.

It shows the measured April click-decline rate across March visibility bands.

It should be interpreted as an observed directional pattern, not evidence that
visibility itself causes decline or that refreshing a page will reverse it.

#### Save decay insight

In [28]:
decay_path = (
    output_dir /
    "w07_decay_refresh_summary.csv"
)

decay_summary.to_csv(
    decay_path,
    index=False
)

print(
    "Decay summary saved:",
    decay_path
)

Decay summary saved: work/outputs/w07_decay_refresh_summary.csv


## Playbook summary

The final playbook ranks observations using the validated Week-5 model probability
and attaches human-readable reason codes and archetypes.

The recommended actions are:

- high visibility + low CTR → review title, snippet, and search intent;
- high traffic + low engagement → review content experience and intent match;
- demand with clicks → review for a possible refresh opportunity;
- visible without a strong issue → human review;
- low evidence → monitor.

The playbook is decision-support.

It does not automatically change content.

The measured model output, reason codes, and retrospective decay analysis should
be reviewed together before any content decision is made.

## Self-check

- [x] Ranked actions are present.
- [x] Every ranked observation has a reason code.
- [x] Archetypes are mapped to recommended actions.
- [x] Intended use is clearly stated.
- [x] Model and playbook limitations are stated.
- [x] Human review is required before action.
- [x] A no-go automation list is included.
- [x] Monitoring triggers are defined.
- [x] Retraining triggers are defined.
- [x] Cost/value thinking is represented through prioritization and review effort.
- [x] A retrospective decay/refresh insight is included.
- [x] The ranked queue is exported to work/outputs/.
- [x] Supporting summaries are exported for the research paper.
- [x] No automatic publishing or content changes are performed.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] No client names, URLs, private queries, or credentials are included.
- [x] The notebook is intended to run top-to-bottom.
- [x] The notebook belongs at work/notebooks/w07_action_playbook.ipynb.

In [29]:
print("ML-10 CONTENT ACTION PLAYBOOK")
print("=" * 60)

print("Queue rows:", len(queue))
print(
    "Mean model probability:",
    round(queue["model_probability"].mean(), 4)
)

print(
    "Human review required:",
    queue["human_review_required"].all()
)

print(
    "Automatic actions allowed:",
    queue["automation_allowed"].any()
)

print("\nExports:")
print(queue_path)
print(archetype_path)
print(monitoring_path)
print(decay_path)

print("\nAssignment 7 notebook checks completed.")

ML-10 CONTENT ACTION PLAYBOOK
Queue rows: 5062
Mean model probability: 0.6687
Human review required: True
Automatic actions allowed: False

Exports:
work/outputs/w07_content_action_queue.csv
work/outputs/w07_archetype_action_summary.csv
work/outputs/w07_monitoring_snapshot.csv
work/outputs/w07_decay_refresh_summary.csv

Assignment 7 notebook checks completed.
